In [3]:
# Michel dos Santos Blanco
# CGU: 331137
# Professor: Bardo Bodmann
# Análise de PRNGs usando a Transformada de Hilbert-Bodmann

import csv
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()

N = 1000
SEED = 12345
DT = 1.0

# Gerador LCG
def lcg(seed, n):
    x = seed
    valores = []
    for _ in range(n):
        x = (1664525 * x + 1013904223) % (2**32)
        valores.append(x / 2**32)
    return np.array(valores)

# Gerador Xorshift32
def xorshift32(seed, n):
    x = seed
    valores = []
    for _ in range(n):
        x ^= (x << 13) & 0xffffffff
        x ^= (x >> 17)
        x ^= (x << 5) & 0xffffffff
        x &= 0xffffffff
        valores.append(x / 2**32)
    return np.array(valores)

# Teste das series 
series = {
    'LCG': lcg(SEED, N),
    'Mersenne Twister': np.random.RandomState(SEED).random_sample(N),
    'Xorshift32': xorshift32(SEED, N)
}

def hilbert_bodmann(u, dt=1.):
    # Aproximação de derivada por diferenças finitas (Equação 1.8)
    n = len(u)
    du = np.gradient(u, dt, edge_order=2)
    v = np.zeros(n)
    
    for i in range(n):
        if i > 0:
            left = np.sum(u[:i][::-1] / np.arange(1, i + 1))
        else:
            left = 0.
            
        if i < n - 1:
            right = np.sum(u[i + 1:] / np.arange(1, n - i))
        else:
            right = 0.
            
        v[i] = (-2 * dt * du[i] + left - right) / np.pi
        
    return v

def analyse(u):
    # Tira a média para centralizar o sinal
    y = u - np.mean(u)
    v = hilbert_bodmann(y)
    
    dy = np.gradient(y, DT, edge_order=2)
    dv = np.gradient(v, DT, edge_order=2)
    
    amp = np.hypot(y, v)
    omega = (y * dv - dy * v) / np.maximum(amp**2, 1e-12)
    freq = omega / (2 * np.pi)
    
    # Separando regiões (centro e bordas)
    lo = N // 8
    hi = 7 * N // 8
    
    edges = np.linspace(-0.5, 0.5, 257)
    
    full, _ = np.histogram(freq, bins=edges, weights=amp)
    central, _ = np.histogram(freq[lo:hi], bins=edges, weights=amp[lo:hi])
    
    borders = np.r_[np.arange(lo), np.arange(hi, N)]
    outer, _ = np.histogram(freq[borders], bins=edges, weights=amp[borders])
    
    # Normalizando os histogramas pela soma dos pesos
    full = full / max(full.sum(), 1e-12)
    central = central / max(central.sum(), 1e-12)
    outer = outer / max(outer.sum(), 1e-12)
    
    centers = (edges[:-1] + edges[1:]) / 2
    low_mask = (np.abs(centers) < 0.05)
    
    lowshare = central[low_mask].sum()
    
    # Calculando autocorrelação para os primeiros 20 lags
    ac_list = []
    for k in range(1, 21):
        r_val = np.corrcoef(u[:-k], u[k:])[0, 1]
        ac_list.append(r_val)
    ac = np.array(ac_list)
    
    acmean = np.mean(np.abs(ac))
    score = 0.5 * lowshare + 0.5 * acmean
    
    return {
        'y': y, 'v': v, 'amp': amp, 'freq': freq,
        'centers': centers, 'full': full, 'central': central, 'outer': outer,
        'lowshare': lowshare, 'ac': ac, 'acmean': acmean, 'score': score
    }

# Rodando a análise para cada gerador 
res = {}
for nome, valores in series.items():
    res[nome] = analyse(valores)

names = list(series.items())
nomes_geradores = list(series.keys())
matriz_series = np.stack([series[k] for k in nomes_geradores])
cor = np.corrcoef(matriz_series)

# Salvando os resultados em CSV 
with open(BASE_DIR / 'resultados.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(['gerador', 'media', 'desvio_padrao', 'fracao_espectral_baixa_frequencia', 'autocorrelacao_media_abs_lags_1_20', 'indice_exploratorio_menor_melhor'])
    for k in nomes_geradores:
        u = series[k]
        r = res[k]
        w.writerow([k, np.mean(u), np.std(u), r['lowshare'], r['acmean'], r['score']])

with open(BASE_DIR / 'correlacoes.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow([''] + nomes_geradores)
    for k, line in zip(nomes_geradores, cor):
        w.writerow([k] + list(line))

with open(BASE_DIR / 'sequencias.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(['i'] + nomes_geradores)
    for i in range(N):
        linha = [i] + [series[k][i] for k in nomes_geradores]
        w.writerow(linha)

# Gerando os gráficos
print("Gerando gráficos...")

# 1. Séries temporais
fig, axs = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
for ax, k in zip(axs, nomes_geradores):
    ax.plot(series[k], lw=0.65)
    ax.set_ylabel(k)
    ax.set_ylim(0, 1)
axs[-1].set_xlabel('Índice da amostra')
fig.suptitle('Sequências pseudo-aleatórias: mesma semente, N=1000')
fig.tight_layout()
fig.savefig(BASE_DIR / '01_sequencias.png', dpi=160)
plt.close(fig)

# 2. Hilbert
fig, axs = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
for ax, k in zip(axs, nomes_geradores):
    r = res[k]
    ax.plot(r['y'][:180], label='Sinal centralizado', lw=0.8)
    ax.plot(r['v'][:180], label='Hilbert Eq. 1.8', lw=0.8)
    ax.set_ylabel(k)
    ax.legend(fontsize=8, loc='upper right')
axs[-1].set_xlabel('Índice da amostra (primeiras 180)')
fig.tight_layout()
fig.savefig(BASE_DIR / '02_hilbert.png', dpi=160)
plt.close(fig)

# 3. Plano complexo
fig, axs = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
for ax, k in zip(axs, nomes_geradores):
    r = res[k]
    ax.scatter(r['y'][:300], r['v'][:300], s=5, alpha=0.45)
    ax.set_ylabel(k + ' / Imaginária')
    ax.set_xlabel('Real')
fig.tight_layout()
fig.savefig(BASE_DIR / '03_plano_complexo.png', dpi=160)
plt.close(fig)

# 4. Frequência instantânea
fig, axs = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
for ax, k in zip(axs, nomes_geradores):
    r = res[k]
    ax.plot(r['freq'], lw=0.55)
    ax.axvspan(125, 875, alpha=0.1)
    ax.set_ylabel(k + ' / ciclos/amostra')
    ax.set_ylim(-0.5, 0.5)
axs[-1].set_xlabel('Índice da amostra')
fig.tight_layout()
fig.savefig(BASE_DIR / '04_frequencia_instantanea.png', dpi=160)
plt.close(fig)

# 5. Espectros
fig, axs = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
for ax, k in zip(axs, nomes_geradores):
    r = res[k]
    ax.plot(r['centers'], r['full'], label='Total', lw=0.9)
    ax.plot(r['centers'], r['central'], label='Central (1/8 a 7/8)', lw=0.9)
    ax.plot(r['centers'], r['outer'], label='Bordas', lw=0.9)
    ax.set_ylabel(k + ' / fração')
    ax.legend(fontsize=7)
axs[-1].set_xlabel('Frequência instantânea assinada (ciclos/amostra)')
fig.tight_layout()
fig.savefig(BASE_DIR / '05_espectros.png', dpi=160)
plt.close(fig)

# Prints finais no console
print('\n--- RESULTADOS DA ANÁLISE ---')
print('Matriz de Correlações:', cor)
for k in nomes_geradores:
    r = res[k]
    print(f"[{k}] Média: {np.mean(series[k]):.4f} | Desvio: {np.std(series[k]):.4f} | Lowshare: {r['lowshare']:.4f} | Autocorr: {r['acmean']:.4f} | Índice: {r['score']:.4f}")

Gerando gráficos...

--- RESULTADOS DA ANÁLISE ---
Matriz de Correlações: [[ 1.          0.02194693 -0.00351286]
 [ 0.02194693  1.         -0.05917601]
 [-0.00351286 -0.05917601  1.        ]]
[LCG] Média: 0.5073 | Desvio: 0.2859 | Lowshare: 0.1160 | Autocorr: 0.0311 | Índice: 0.0735
[Mersenne Twister] Média: 0.5046 | Desvio: 0.2878 | Lowshare: 0.0954 | Autocorr: 0.0191 | Índice: 0.0573
[Xorshift32] Média: 0.4885 | Desvio: 0.2919 | Lowshare: 0.0687 | Autocorr: 0.0259 | Índice: 0.0473
